## module 2 — subject segmentation

distilled binary segmenter (plant vs. background). the teacher is FastSAM, run offline over a stratified 20K subset of the train split by `scripts/generate_pseudo_masks.py`; the student here is **DeepLabV3 + MobileNetV3-Large** (~11 MB on disk), light enough to ride the demo pipeline as a soft-mask preprocessor before the ResNet-50 classifier. **IoU on val** is the direct metric; **Δtop-1 on the classifier** (measured by `scripts/measure_delta_top1.py`) is the downstream one.

**you only customize two cells** — the *paired augmentation* and the *model / optimizer / schedule*. all the plumbing (data loading, VRAM-aware batching, resumable checkpointing across local · colab · kaggle, the training loop) lives in `bvtrain/`.

runs anywhere; resumable. before running, either publish the masks dataset (`python scripts/generate_pseudo_masks.py --repo <you>/botanical-vision-256-masks`) or point `hf_masks_repo` below at an existing one.

In [ ]:
# get bvtrain (shared training plumbing): locally it's ../bvtrain; on colab/kaggle we clone the repo
import os, sys
_CANDS = ["..", ".", "botanical-vision"]
if not any(os.path.isdir(f"{p}/bvtrain") for p in _CANDS):
    os.system("git clone -q https://github.com/babnigg/botanical-vision.git")
for _p in _CANDS:
    if os.path.isdir(f"{_p}/bvtrain"):
        sys.path.insert(0, _p)
        break

import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
import bvtrain as bv

In [ ]:
# point env at your masks dataset (either published by you or a shared one).
# either edit the placeholder below or export BV_MASKS_REPO before starting the kernel.
env = bv.setup(hf_masks_repo=os.environ.get("BV_MASKS_REPO", "<your-user>/botanical-vision-256-masks"))

In [ ]:
data = bv.load_seg_data(env)

In [ ]:
# ── customize: paired augmentation ──
# spatial ops MUST apply to image AND mask together (same crop, same flip);
# color ops apply to image only. no RandomErasing — it would delete the
# subject we're trying to learn to segment.
def train_paired(img, mask):
    i, j, h, w = transforms.RandomResizedCrop.get_params(img, scale=(0.6, 1.0), ratio=(0.9, 1.1))
    img  = TF.resized_crop(img,  i, j, h, w, [bv.IMG_SIZE, bv.IMG_SIZE], InterpolationMode.BILINEAR)
    mask = TF.resized_crop(mask, i, j, h, w, [bv.IMG_SIZE, bv.IMG_SIZE], InterpolationMode.NEAREST)
    if torch.rand(1).item() < 0.5:
        img, mask = TF.hflip(img), TF.hflip(mask)
    img = transforms.ColorJitter(0.3, 0.3, 0.3)(img)
    img = TF.to_tensor(img)
    img = TF.normalize(img, bv.MEAN, bv.STD)
    m = torch.from_numpy(np.array(mask, dtype=np.uint8))
    m = (m > 127).float().unsqueeze(0)
    return img, m

In [ ]:
# ── customize: model, optimizer, schedule ──
EPOCHS = 10
# pretrained backbone + head (21 COCO classes); we swap the final 1x1 conv to 1 output logit
model = deeplabv3_mobilenet_v3_large(weights="DEFAULT", aux_loss=False)
in_ch = model.classifier[-1].in_channels
model.classifier[-1] = nn.Conv2d(in_ch, 1, kernel_size=1)
model = model.to(env.device)

# discriminative lr: pretrained backbone learns slowly, fresh head faster
backbone = [p for n, p in model.named_parameters() if not n.startswith("classifier.")]
head     = [p for n, p in model.named_parameters() if n.startswith("classifier.")]
optimizer = torch.optim.AdamW(
    [{"params": backbone, "lr": 1e-4}, {"params": head, "lr": 1e-3}],
    weight_decay=1e-2,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
loaders = bv.build_seg_loaders(data, train_paired, env)
hist = bv.fit_seg(model, optimizer, loaders, epochs=EPOCHS, run_name="deeplabv3_mnv3_seg",
                  env=env, scheduler=scheduler)

In [ ]:
bv.plot_seg_history(hist)
bv.evaluate_seg(model, loaders.test, env, run_name="deeplabv3_mnv3_seg")

# qualitative check: image | pred mask | teacher (pseudo) mask
import matplotlib.pyplot as plt
model.eval()
n = 6
fig, axs = plt.subplots(n, 3, figsize=(9, 3 * n))
fig.set_facecolor("linen")
mean = torch.tensor(bv.MEAN).view(3, 1, 1)
std  = torch.tensor(bv.STD).view(3, 1, 1)
with torch.no_grad():
    for k in range(n):
        x, y = loaders.test.dataset[k]
        pred = torch.sigmoid(model(x.unsqueeze(0).to(env.device))["out"])[0, 0].cpu()
        img = (x * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axs[k, 0].imshow(img);                          axs[k, 0].set_title("image");   axs[k, 0].axis("off")
        axs[k, 1].imshow(pred, cmap="viridis", vmin=0, vmax=1); axs[k, 1].set_title("pred"); axs[k, 1].axis("off")
        axs[k, 2].imshow(y[0], cmap="gray");            axs[k, 2].set_title("teacher"); axs[k, 2].axis("off")
plt.tight_layout()
plt.show()